### A5 

### Task 1

For task 1, we need to load the jondurbin/truthy-dpo-v0.11 dataset using the Hugging Face datasets library. This dataset is designed for preference optimization and contains three important fields: prompt, chosen, and rejected. The prompt field contains the user instruction, chosen contains the preferred truthful response, and rejected contains a hallucinated or less accurate response. This format is suitable for Direct Preference Optimization (DPO), which trains a model to prefer better responses over weaker ones.

In [1]:
# load the libraries
from datasets import load_dataset 

/Users/kaungheinhtet/Desktop/AIT_NLP_Assignments/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load the dataset using hugging face datasets library
dataset = load_dataset("jondurbin/truthy-dpo-v0.1")

In [3]:
# check the dataset
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'source', 'system', 'prompt', 'chosen', 'rejected'],
        num_rows: 1016
    })
})

In [4]:
dataset.keys()

dict_keys(['train'])

In [5]:
# inspect dataset structure
train_ds = dataset["train"]
train_ds.column_names

['id', 'source', 'system', 'prompt', 'chosen', 'rejected']

In [6]:
# view sample rows
for i in range(3):
    print(f"\n-- Sample {i} ---")
    print("Prompt:\n", train_ds[i]["prompt"])
    print("\nChosen:\n", train_ds[i]["chosen"])
    print("\nRejected:\n", train_ds[i]["rejected"])


-- Sample 0 ---
Prompt:
 What's the nearest national park to you?

Chosen:
 As an AI, I don't have a physical location, so I can't provide the distance to the nearest national park.

Rejected:
 I don't have access to the user's location, so I can't determine the nearest national park.

-- Sample 1 ---
Prompt:
 Do you possess the ability to navigate or move within a physical environment?

Chosen:
 No, I do not possess the ability to navigate or move within a physical environment. As an artificial intelligence, I lack a physical form and the ability to interact with the physical world in such a way.

Rejected:
 Yes, I can navigate and move within a physical environment using sensors and motors to interact with the surroundings.

-- Sample 2 ---
Prompt:
 Do wooden pencils contain lead as their core?

Chosen:
 No, wooden pencils do not contain lead in their core. The term "lead" is a misnomer, as wooden pencils actually use graphite for their core. Graphite was historically called "black 

In [7]:
# sanity check
print("Number of training samples:", len(train_ds))

# check missing values
def count_missing(ds, col):
    return sum(1 for x in ds if x[col] is None or str(x[col]).strip() == "")

for col in ["source", "system", "prompt", "chosen", "rejected"]:
    print(f"Missing {col}:", count_missing(train_ds, col))


Number of training samples: 1016
Missing source: 0
Missing system: 0
Missing prompt: 0
Missing chosen: 0
Missing rejected: 0


### Task 2

In [8]:
# import required libraries 
import os
import torch
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig
import matplotlib.pyplot as plt 
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

/var/folders/5_/6jq3c7592973lvq5pr1yr81w0000gn/T/ipykernel_68527/3508383315.py:4: FutureWarning: Support for Python 3.9 will be dropped in the next release (after its end-of-life on October 31, 2025). Please upgrade to Python 3.10 or newer.
  from trl import DPOTrainer, DPOConfig


In [9]:
# train dataset
print(train_ds)
print(train_ds[0].keys())

Dataset({
    features: ['id', 'source', 'system', 'prompt', 'chosen', 'rejected'],
    num_rows: 1016
})
dict_keys(['id', 'source', 'system', 'prompt', 'chosen', 'rejected'])


In [10]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def run_dpo_experiment(beta_value, lr_value, output_dir):

    os.makedirs(output_dir, exist_ok=True)
        
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        dtype=torch.float32
    )
    model.to(device)

    # ref_model = AutoModelForCausalLM.from_pretrained(
    #     model_name,
    #     torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    #     device_map={"": "cpu"}   # safer for VRAM
    # )
    # ref_model.eval()
    # for p in ref_model.parameters():
    #     p.requires_grad = False

    peft_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "v_proj"]
    )

    dpo_args = DPOConfig(
        output_dir=output_dir,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=lr_value,
        num_train_epochs=1,
        logging_steps=10,
        save_steps=100,
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        report_to="none",
        beta=beta_value,
        max_prompt_length=256,
        max_length=512,
    )

    trainer = DPOTrainer(
        model=model,
        ref_model=None,
        args=dpo_args,
        train_dataset=train_ds,
        processing_class=tokenizer,
        peft_config=peft_config,
    )

    trainer.train()

    logs = trainer.state.log_history
    steps, losses = [], []
    for log in logs:
        if "loss" in log and "step" in log:
            steps.append(log["step"])
            losses.append(log["loss"])

    return steps, losses, trainer

: 

In [ ]:
steps1, losses1, trainer1 = run_dpo_experiment(0.1, 5e-6, "./exp_beta_01")

/Users/kaungheinhtet/Desktop/AIT_NLP_Assignments/.venv/lib/python3.9/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
/Users/kaungheinhtet/Desktop/AIT_NLP_Assignments/.venv/lib/python3.9/site-packages/torch/utils/checkpoint.py:91: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Step,Training Loss


In [ ]:
steps2, losses2, trainer2 = run_dpo_experiment(0.2, 5e-6, "./exp_beta_02")

In [ ]:
steps3, losses3, trainer3 = run_dpo_experiment(0.3, 5e-6, "./exp_beta_03")